
# Machine Learning Forecasting Models

This notebook evaluates XGBoost for one-month-ahead bilateral trade-flow forecasting. A global model is trained across product-partner series for each reporting country, using historical lag, rolling, and calendar features.

Training uses observations through February 2026, with March 2026 held out for evaluation. Forecasts are evaluated on the original trade-value scale using MAE, RMSE, and sMAPE.


## 1. Load Forecasting Data

Load the prepared training data and held-out March 2026 observations for each country.

In [5]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_loading import (
    load_forecasting_data,
    load_feature_data,
    SERIES_KEYS,
    TARGET,
)
from src.evaluation import evaluate_forecasts, save_model_results

training_data, test_data = load_forecasting_data()
feature_data = load_feature_data()

TEST_MONTH = pd.Timestamp("2026-03-01")

print("Training datasets:")
for country, df in training_data.items():
    print(f"{country}: {len(df):,} rows")

print("\nTest datasets:")
for country, df in test_data.items():
    print(f"{country}: {len(df):,} rows")


Training datasets:
South Korea: 2,374,832 rows
Slovakia: 2,826,210 rows
USA: 3,880,587 rows

Test datasets:
South Korea: 27,823 rows
Slovakia: 21,973 rows
USA: 35,843 rows




## 2. Feature Selection and Target Preparation

Use the existing lag, rolling, and calendar features to predict `Value` directly. Exclude identifiers, dates, and target-derived information from the predictors. Use a time-based validation split for hyperparameter selection, keeping March 2026 untouched until final evaluation.



In [6]:

LAG_FEATURES = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_12",
]

ROLLING_FEATURES = [
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_mean_12",
    "rolling_std_3",
    "rolling_std_6",
    "rolling_std_12",
]

CALENDAR_FEATURES = [
    "month_of_year",
    "month_sin",
    "month_cos",
]

FEATURE_COLUMNS = LAG_FEATURES + ROLLING_FEATURES + CALENDAR_FEATURES

XGBOOST_CONFIGS = {
    "XGBoost_1": {
        "n_estimators": 200,
        "max_depth": 4,
        "learning_rate": 0.05,
    },
    "XGBoost_2": {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
    },
    "XGBoost_3": {
        "n_estimators": 300,
        "max_depth": 8,
        "learning_rate": 0.03,
    },
}

print(f"Number of features: {len(FEATURE_COLUMNS)}")
print(f"Configurations defined: {len(XGBOOST_CONFIGS)}")


Number of features: 14
Configurations defined: 3



## 3.0 Training and Validation Split

The training data are split chronologically, using observations before February 2026 for model fitting and February 2026 for validation. Rows with missing feature or target values are excluded. March 2026 remains reserved for the final evaluation.

This validation set will be used to compare the three predefined XGBoost configurations using MAE, without using the final test results for model selection.


In [7]:

import pandas as pd
VALIDATION_MONTH = pd.Timestamp("2026-02-01")

validation_training_data = {}
validation_data = {}

required_columns = FEATURE_COLUMNS + [TARGET, "Month"]

for country, df in training_data.items():
    missing_columns = set(required_columns) - set(df.columns)
    if missing_columns:
        raise ValueError(
            f"{country} is missing columns: {sorted(missing_columns)}"
        )

    train_df = df.loc[df["Month"] < VALIDATION_MONTH].copy()
    val_df = df.loc[df["Month"] == VALIDATION_MONTH].copy()

    train_df = train_df.dropna(subset=required_columns)
    val_df = val_df.dropna(subset=required_columns)

    validation_training_data[country] = train_df
    validation_data[country] = val_df

    print(
        f"{country}: {len(train_df):,} training rows; "
        f"{len(val_df):,} validation rows"
    )


South Korea: 2,013,133 training rows; 27,823 validation rows
Slovakia: 2,540,561 training rows; 21,973 validation rows
USA: 3,414,628 training rows; 35,843 validation rows



## 4.0 XGBoost Configuration Selection

Train one global XGBoost model per reporting country for each of the three predefined configurations. Compare their February 2026 validation MAE and select the best configuration for each country. March 2026 remains untouched during this selection process.


In [8]:

validation_results = []
validation_models = {}

for country, train_df in validation_training_data.items():
    val_df = validation_data[country]

    X_train = train_df[FEATURE_COLUMNS]
    y_train = train_df[TARGET]

    X_val = val_df[FEATURE_COLUMNS]
    y_val = val_df[TARGET]

    for config_name, params in XGBOOST_CONFIGS.items():
        model = XGBRegressor(
            **params,
            objective="reg:squarederror",
            tree_method="hist",
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        predictions = np.maximum(model.predict(X_val), 0.0)
        
        actual = y_val.to_numpy()
        absolute_errors = np.abs(actual - predictions)

        mae = absolute_errors.mean()
        rmse = np.sqrt(np.mean((actual - predictions) ** 2))

        denominator = np.abs(actual) + np.abs(predictions)
        smape_terms = np.divide(
            2 * absolute_errors,
            denominator,
            out=np.zeros_like(absolute_errors, dtype=float),
            where=denominator != 0,
        )
        smape = smape_terms.mean() * 100
        validation_results.append({
        "Country": country,
        "Configuration": config_name,
        "Validation Month": "2026-02",
        "N Series": len(val_df),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "sMAPE (%)": float(smape),
        })


        validation_models[(country, config_name)] = model

validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values(["Country", "MAE"])
    .reset_index(drop=True)
)

display(validation_results_df)


,Country,Configuration,Validation Month,N Series,MAE,RMSE,sMAPE (%)
0,Slovakia,XGBoost_1,2026-02,21973,2.917972e+05,5.573460e+06,93.649317
1,Slovakia,XGBoost_3,2026-02,21973,2.953965e+05,5.477924e+06,88.848252
2,Slovakia,XGBoost_2,2026-02,21973,2.997424e+05,5.476564e+06,89.573324
3,South Korea,XGBoost_3,2026-02,27823,1.645711e+06,3.740413e+07,107.989183
4,South Korea,XGBoost_2,2026-02,27823,1.651236e+06,3.787066e+07,107.770272
5,South Korea,XGBoost_1,2026-02,27823,1.724899e+06,3.766602e+07,117.194067
6,USA,XGBoost_3,2026-02,35843,3.911390e+06,9.177904e+07,82.615534
7,USA,XGBoost_1,2026-02,35843,3.923460e+06,9.111577e+07,89.802823
8,USA,XGBoost_2,2026-02,35843,3.924299e+06,9.182718e+07,82.414293



## 5. Select the Best Configuration

For each reporting country, select the XGBoost configuration with the lowest February 2026 validation MAE. The selected configuration will then be refitted using all eligible training observations through February 2026 before generating the March forecasts.


In [9]:

best_configurations = (
    validation_results_df
    .sort_values(["Country", "MAE"])
    .drop_duplicates(subset="Country")
    .set_index("Country")["Configuration"]
    .to_dict()
)

print("Selected XGBoost configuration per country:")
for country, config_name in best_configurations.items():
    print(f"{country}: {config_name}")


Selected XGBoost configuration per country:
Slovakia: XGBoost_1
South Korea: XGBoost_3
USA: XGBoost_3



## 6.Final March 2026 Forecasts

Refit the selected XGBoost configuration for each country using all eligible observations through February 2026. Generate non-negative predictions for March 2026 using the corresponding March feature rows, then evaluate them against the held-out actual values using MAE, RMSE, and sMAPE.


In [14]:

final_predictions = {}
final_models = {}
xgboost_results = []

for country, train_df in training_data.items():
    config_name = best_configurations[country]
    params = XGBOOST_CONFIGS[config_name]

    # Retrieve March 2026 features and actual targets
    march_features = feature_data[country].loc[
        feature_data[country]["Month"] == TEST_MONTH
    ].copy()

    test_df = test_data[country].copy()

    # Remove any existing target columns before merging actual values
    march_features = march_features.drop(
        columns=["Value", "Value_x", "Value_y"],
        errors="ignore",
    )

    march_features = march_features.merge(
        test_df[SERIES_KEYS + [TARGET]],
        on=SERIES_KEYS,
        how="inner",
        validate="one_to_one",
    )

    # Fit the selected configuration on all data through February 2026
    model = XGBRegressor(
        **params,
        objective="reg:squarederror",
        tree_method="hist",
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(
        train_df[FEATURE_COLUMNS],
        train_df[TARGET],
    )

    # Forecast March on the original trade-value scale
    predictions = np.maximum(
        model.predict(march_features[FEATURE_COLUMNS]),
        0.0,
    )

    actual = march_features[TARGET].to_numpy()

    final_predictions[country] = march_features[
        SERIES_KEYS + [TARGET]
    ].copy()
    final_predictions[country]["xgboost_prediction"] = predictions
    final_models[country] = model

    xgboost_results.append(
        evaluate_forecasts(
            country=country,
            model_name=f"XGBoost ({config_name})",
            actual=actual,
            predicted=predictions,
            forecast_month="2026-03",
        )
    )

# Save the results for all three countries
xgboost_results_df = save_model_results(
    xgboost_results,
    model_name="XGBoost",
)

display(xgboost_results_df)



Saved model results to: C:\Transfer\InterdisciplinaryProject\results\xgboost_results.csv


,Country,Model,Forecast Month,N Series,MAE,RMSE,sMAPE (%)
0,Slovakia,XGBoost (XGBoost_1),2026-03,21973,3.548070e+05,5.700961e+06,88.860391
1,South Korea,XGBoost (XGBoost_3),2026-03,27823,2.013895e+06,5.278073e+07,101.690338
2,USA,XGBoost (XGBoost_3),2026-03,35843,4.326257e+06,9.880096e+07,79.029049



## Summary

This notebook evaluated three predefined XGBoost configurations for one-month-ahead bilateral trade-flow forecasting for South Korea, Slovakia, and the USA. The configurations were compared using February 2026 validation MAE, RMSE, and sMAPE, and the configuration with the lowest validation MAE was selected separately for each country.

The selected models were refitted using all eligible observations through February 2026 and used to forecast March 2026. Predictions were evaluated on the original trade-value scale using MAE, RMSE, and sMAPE. Results were saved to the `results/` directory for the final comparison with the baseline and statistical models.

An exhaustive grid search was not performed due to computational limitations. Instead, three predefined configurations were evaluated. The single-month validation and test periods also limit how broadly the results can be generalized.
